In [1]:
import sys, time, csv, itertools, zipfile
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from pyspark.sql import functions as F, types as T
from src.config import load_config
from src.spark_session import get_spark

RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed" / "large"
BEH_COLS = ["impression_id", "user_id", "time", "history", "impressions"]
W, ALPHA = 0.6, 20            # fijados en 09b (validación sin clics recientes)

t0 = time.time()
spark = get_spark(load_config("small"))
schema = T.StructType([T.StructField(c, T.StringType()) for c in BEH_COLS])
beh = (spark.read.option("sep", "\t").option("header", False).option("quote", "\u0000")
            .schema(schema)
            .csv([str(RAW / "MINDlarge_train" / "behaviors.tsv"),
                  str(RAW / "MINDlarge_dev" / "behaviors.tsv")]))

ctr_df = (beh.select(F.explode(F.split("impressions", " ")).alias("item"))
             .select(F.split("item", "-")[0].alias("news_id"),
                     F.split("item", "-")[1].cast("int").alias("label"))
             .groupBy("news_id")
             .agg(F.sum("label").alias("clics"), F.count("*").alias("expos"))
             .toPandas())
spark.stop()

ctr_df.to_parquet(OUT / "ctr_estatico_large.parquet", index=False)
p0 = ctr_df["clics"].sum() / ctr_df["expos"].sum()
print(f"Noticias con CTR: {len(ctr_df):,} | p0 = {p0:.4f} | {time.time() - t0:.0f} s")

Noticias con CTR: 29,309 | p0 = 0.0406 | 34 s


In [2]:
emb = pd.read_parquet(OUT / "news_emb_minilm_all.parquet")
idx = {n: i for i, n in enumerate(emb["news_id"])}
E = np.stack(emb["emb"].to_numpy()).astype(np.float32)
del emb

ctr_vec = np.full(len(E), p0, dtype=np.float64)           # sin exposiciones -> p0
tiene_ctr = np.zeros(len(E), dtype=bool)
pos = ctr_df["news_id"].map(idx)
ok = pos.notna()
p = pos[ok].astype(int).to_numpy()
ctr_vec[p] = (ctr_df.loc[ok, "clics"] + ALPHA * p0) / (ctr_df.loc[ok, "expos"] + ALPHA)
tiene_ctr[p] = True
print(f"Embeddings: {E.shape} | noticias del catálogo con CTR: {tiene_ctr.sum():,}")

Embeddings: (130379, 384) | noticias del catálogo con CTR: 29,309


In [3]:
def z_por_grupo(x, g, n):
    """Puntaje z dentro de cada impresión (g = índice de impresión en el bloque)."""
    cnt = np.bincount(g, minlength=n)
    mu = np.bincount(g, x, n) / cnt
    sd = np.sqrt(np.maximum(np.bincount(g, x * x, n) / cnt - mu ** 2, 0))
    s = sd[g]
    return np.where(s > 1e-12, (x - mu[g]) / np.where(s > 1e-12, s, 1.0), 0.0)

rng = np.random.default_rng(42)
pred_path = OUT / "prediction.txt"
lector = pd.read_csv(RAW / "MINDlarge_test" / "behaviors.tsv", sep="\t", header=None,
                     names=BEH_COLS, quoting=csv.QUOTE_NONE, dtype=str, chunksize=10_000)

t0 = time.time()
n_imp = n_cand = n_sin_ctr = 0
with open(pred_path, "w", encoding="utf-8", newline="\n") as out:
    for b, chunk in enumerate(lector):
        n = len(chunk)
        # Perfil por impresión = suma de embeddings del historial (la escala no afecta al z-score)
        h_idx = [[idx[x] for x in h.split() if x in idx] for h in chunk["history"].fillna("")]
        lens_h = np.fromiter((len(h) for h in h_idx), dtype=np.int64, count=n)
        P = np.zeros((n, E.shape[1]), dtype=np.float32)
        flat_h = np.fromiter(itertools.chain.from_iterable(h_idx), dtype=np.int64)
        if flat_h.size:
            offs = np.concatenate([[0], np.cumsum(lens_h)[:-1]])
            P[lens_h > 0] = np.add.reduceat(E[flat_h], offs[lens_h > 0], axis=0)

        # Candidatas
        c_idx = [[idx[x] for x in c.split()] for c in chunk["impressions"]]
        lens_c = np.fromiter((len(c) for c in c_idx), dtype=np.int64, count=n)
        flat_c = np.fromiter(itertools.chain.from_iterable(c_idx), dtype=np.int64)
        g = np.repeat(np.arange(n), lens_c)

        cos = np.einsum("ij,ij->i", E[flat_c], P[g]).astype(np.float64)
        cos = np.where(lens_h[g] > 0, cos, 0.0)          # sin historial: constante -> z = 0
        score = W * z_por_grupo(cos, g, n) + (1 - W) * z_por_grupo(ctr_vec[flat_c], g, n)

        # Ranking dentro de cada impresión (1 = mejor), desempate aleatorio con semilla fija
        order = np.lexsort((rng.random(len(score)), -score, g))
        starts = np.concatenate([[0], np.cumsum(lens_c)[:-1]])
        rank = np.empty(len(order), dtype=np.int64)
        rank[order] = np.arange(len(order)) - starts[g[order]] + 1

        partes = np.split(rank, np.cumsum(lens_c)[:-1])
        if b == 0:   # comprobación: cada lista es una permutación de 1..N
            assert all(np.array_equal(np.sort(r), np.arange(1, len(r) + 1)) for r in partes)
        out.write("\n".join(f"{iid} [{','.join(map(str, r))}]"
                            for iid, r in zip(chunk["impression_id"], partes)) + "\n")

        n_imp += n; n_cand += len(flat_c); n_sin_ctr += int((~tiene_ctr[flat_c]).sum())
        if b % 20 == 0:
            print(f"  {n_imp:>9,} impresiones ({time.time() - t0:5.0f} s)")

print(f"\nImpresiones: {n_imp:,} | candidatas: {n_cand:,} | "
      f"candidatas sin CTR: {n_sin_ctr / n_cand:.1%} | {(time.time() - t0) / 60:.1f} min")

     10,000 impresiones (    5 s)
    210,000 impresiones (   92 s)
    410,000 impresiones (  179 s)
    610,000 impresiones (  262 s)
    810,000 impresiones (  480 s)
  1,010,000 impresiones (  652 s)
  1,210,000 impresiones (  817 s)
  1,410,000 impresiones (  956 s)
  1,610,000 impresiones ( 1139 s)
  1,810,000 impresiones ( 1278 s)
  2,010,000 impresiones ( 1431 s)
  2,210,000 impresiones ( 1539 s)

Impresiones: 2,370,727 | candidatas: 93,115,001 | candidatas sin CTR: 83.3% | 26.6 min


In [4]:
with open(pred_path, encoding="utf-8") as fh:
    n_lineas = sum(1 for _ in fh)
print(f"Líneas: {n_lineas:,} (deben ser 2,370,727)")
with open(pred_path, encoding="utf-8") as fh:
    for _ in range(3):
        print(fh.readline().strip()[:120])
print(f"Tamaño: {pred_path.stat().st_size / 1e6:.0f} MB")

Líneas: 2,370,727 (deben ser 2,370,727)
1 [4,13,7,14,9,15,16,6,3,1,11,5,2,10,12,8]
2 [2,5,1,6,4,7,3]
3 [46,66,4,7,29,63,76,72,20,75,23,56,35,21,53,19,61,37,43,40,38,57,41,13,62,45,69,26,81,60,15,77,49,44,48,52,34,17,2,25,
Tamaño: 291 MB


In [5]:
zip_path = OUT / "prediction.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname="prediction.txt")     # sin carpetas dentro del zip
print(f"{zip_path} ({zip_path.stat().st_size / 1e6:.0f} MB)")

c:\DevProjects\MMDS\RecSysMIND\data\processed\large\prediction.zip (108 MB)
